## Purpose:
Identify which Fingertips indicators are relevant to health inequalities in North West London before pulling any data.

## What this notebook does:

- Downloads the Fingertips indicator metadata catalogue

- Inspects indicator names, definitions, and content

- Filters indicators by clinical areas (e.g. maternity, mental health, diabetes)

- Produces a shortlist of indicator IDs for ingestion

## Output:

_shortlist_fingertips_candidates.csv
(the contract input for Notebook 02)

## One-line summary:

- Decide what data to use before deciding how to analyse it.

In [18]:
import requests          # allows Python talk to the web (Python's web downloader)
from pathlib import Path # Path manages files and folders on the machine 

HERE = Path.cwd()
METADATA_URL = "https://fingertips.phe.org.uk/api/indicator_metadata/csv/all"

final_path = HERE / "_raw_fingertips_indicator_metadata.csv" # is used when download is 100% complete
tmp_path = HERE / "_raw_fingertips_indicator_metadata.tmp"   # is used while downloading in case smth fails 

with requests.get(METADATA_URL, stream=True, timeout=120) as r:
    r.raise_for_status()
    with open(tmp_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

tmp_size = tmp_path.stat().st_size
print("Downloaded tmp size (bytes):", tmp_size)

tmp_path.replace(final_path)
print("Final size (bytes):", final_path.stat().st_size)

Downloaded tmp size (bytes): 6706808
Final size (bytes): 6706808


In [19]:
import pandas as pd

meta_head = pd.read_csv(final_path, nrows=5)
print(meta_head.columns.tolist())
print(meta_head.shape)

['Indicator ID', 'Indicator', 'Indicator number', 'Rationale', 'Specific rationale', 'Definition', 'Data source', 'Indicator source', 'Definition of numerator', 'Source of numerator', 'Definition of denominator', 'Source of denominator', 'Methodology', 'Standard population/values', 'Frequency', 'Confidence interval details', 'Disclosure control', 'Rounding', 'Caveats', 'Notes', 'Impact of COVID-19', 'Copyright', 'Data re-use', 'Links', 'Indicator Content', 'Simple Name', 'Simple Definition', 'Unit', 'Value type', 'Year type', 'Polarity', 'Date updated']
(5, 32)


In [20]:
meta = pd.read_csv(final_path, low_memory=False)
print(meta.shape)

(1008, 32)


In [21]:
meta.to_parquet("_processed_fingertips_indicator_metadata.parquet", index=False, engine="pyarrow")

In [22]:
meta.columns

Index(['Indicator ID', 'Indicator', 'Indicator number', 'Rationale',
       'Specific rationale', 'Definition', 'Data source', 'Indicator source',
       'Definition of numerator', 'Source of numerator',
       'Definition of denominator', 'Source of denominator', 'Methodology',
       'Standard population/values', 'Frequency',
       'Confidence interval details', 'Disclosure control', 'Rounding',
       'Caveats', 'Notes', 'Impact of COVID-19', 'Copyright', 'Data re-use',
       'Links', 'Indicator Content', 'Simple Name', 'Simple Definition',
       'Unit', 'Value type', 'Year type', 'Polarity', 'Date updated'],
      dtype='str')

In [23]:
meta.head(5)

,Indicator ID,Indicator,Indicator number,Rationale,Specific rationale,Definition,Data source,Indicator source,Definition of numerator,Source of numerator,...,Data re-use,Links,Indicator Content,Simple Name,Simple Definition,Unit,Value type,Year type,Polarity,Date updated
0,108,Under 75 mortality rate from all causes,NaN,Premature mortality is a good high-level indic...,NaN,Directly age-standardised mortality rate for a...,"OHID, based on Office for National Statistics ...",Office for Health Improvement and Disparities,Number of deaths registered in the respective ...,"Office for National Statistics (ONS), Annual m...",...,Please cite any use of this website as follows...,NaN,Directly age-standardised rate of mortality fr...,"Deaths from all causes, ages under 75",Rate of deaths from all causes for people aged...,"per 100,000",Directly standardised rate,Calendar,RAG - Low is good,13/11/2025
1,114,QOF Total List Size,NaN,NaN,NaN,Total number of patients registered with the p...,"NHS England, Quality and Outcomes Framework",For all QOF data for the current and for previ...,NaN,"NHS England (NHSE), Quality and Outcomes Frame...",...,NaN,NaN,NaN,NaN,NaN,NaN,Count,Financial,Not applicable,10/12/2025
2,200,Learning disability: QOF prevalence,NaN,NaN,NaN,The percentage of patients with learning disab...,NHS England,For all QOF data for the current and for previ...,Total number of patients with learning disabil...,"NHS England (NHSE), Quality and Outcomes Frame...",...,NaN,NaN,NaN,NaN,NaN,%,Proportion,Financial,BOB - Blue orange blue,10/12/2025
3,212,Stroke: QOF prevalence,NaN,Stroke is the third most common cause of death...,NaN,The percentage of patients with stroke or tran...,NHS England,For all QOF data for the current and for previ...,Patients with stroke or transient ischaemic at...,"NHS England (NHSE), Quality and Outcomes Frame...",...,NaN,NaN,NaN,Stroke prevalence,Percentage of patients with stroke or transien...,%,Proportion,Financial,BOB - Blue orange blue,10/12/2025
4,219,Hypertension: QOF prevalence,NaN,NaN,NaN,The percentage of patients with established hy...,NHS England,For all QOF data for the current and for previ...,The number of patients with established hypert...,"NHS England (NHSE), Quality and Outcomes Frame...",...,NaN,NaN,NaN,Hypertension prevalence,Percentage of patients with established hypert...,%,Proportion,Financial,BOB - Blue orange blue,10/12/2025


In [24]:
import re
import pandas as pd

SEARCH_COLS = ["Indicator", "Simple Name", "Indicator Content", "Definition"]
SEARCH_COLS = [c for c in SEARCH_COLS if c in meta.columns]

combined = (
    meta[SEARCH_COLS]
    .fillna("")
    .astype(str)
    .agg(" | ".join, axis=1)
    .str.lower()
)

KEYWORDS = {
    "maternity": [r"\bmaternity\b", r"\bpregnan\w*\b", r"\bantenatal\b", r"\bpostnatal\b",
                 r"\blow birth weight\b", r"\bstillbirth\b", r"\bneonatal\b", r"\bperinatal\b",
                 r"\bmidwif\w*\b", r"\blabou?r\b", r"\bbirth\b"],
    "mental_health": [r"\bmental\b", r"\bsuicide\b", r"\bself[- ]harm\b", r"\bpsychos\w*\b",
                      r"\bdepress\w*\b", r"\banxiet\w*\b", r"\bsmi\b", r"\bschizophren\w*\b"],
    "diabetes": [r"\bdiabetes\b", r"\bhba1c\b", r"\bglyc\w*\b"],
    "respiratory_asthma": [r"\basthma\b", r"\bcopd\b", r"\brespirat\w*\b", r"\blung\b", r"\bronch\w*\b"],
    "hypertension_cvd": [r"\bhypertension\b", r"\bblood pressure\b", r"\bstroke\b", r"\bcardio\w*\b", r"\bheart\b", r"\bcvd\b"],
    "early_cancer_dx": [r"\bcancer\b", r"\bscreening\b", r"\bdiagnos\w*\b", r"\bstage\b"],
    "epilepsy": [r"\bepilepsy\b", r"\bseizure\b"],
}

rows = []
for theme, patterns in KEYWORDS.items():
    pattern = "|".join(patterns)
    mask = combined.str.contains(pattern, regex=True, na=False)

    tmp = meta.loc[mask, ["Indicator ID", "Indicator", "Simple Name", "Definition", "Data source", "Date updated"]].copy()
    tmp["match_theme"] = theme
    tmp["match_terms"] = pattern
    rows.append(tmp)

shortlist = (
    pd.concat(rows, ignore_index=True)
    .drop_duplicates(subset=["Indicator ID", "match_theme"])
)

shortlist["primary_theme"] = shortlist["match_theme"]
shortlist["keep"] = True

print("Shortlist rows:", len(shortlist))
shortlist.sort_values(["match_theme", "Indicator"]).head(20)

Shortlist rows: 501


,Indicator ID,Indicator,Simple Name,Definition,Data source,Date updated,match_theme,match_terms,primary_theme,keep
128,241,Diabetes: QOF prevalence,Diabetes prevalence,The percentage of patients aged 17 or over wit...,NHS England,10/12/2025,diabetes,\bdiabetes\b|\bhba1c\b|\bglyc\w*\b,diabetes,True
165,93347,Estimated diabetes diagnosis rate,NaN,"The estimated diabetes diagnosis rate, express...",Office for Health Improvement and Disparities,28/01/2019,diabetes,\bdiabetes\b|\bhba1c\b|\bglyc\w*\b,diabetes,True
185,94192,Estimated prevalence of diagnosed and undiagno...,Total type 2 diabetes prevalence,Modelled estimated prevalence of diagnosed and...,"OHID, based on Office for National Statistics ...",25/11/2025,diabetes,\bdiabetes\b|\bhba1c\b|\bglyc\w*\b,diabetes,True
135,92622,Hospital admissions for diabetes (under 19 years),NaN,Emergency admissions for diabetes for children...,"OHID, based on NHS England and Office for Nati...",30/04/2025,diabetes,\bdiabetes\b|\bhba1c\b|\bglyc\w*\b,diabetes,True
164,93051,Hospital spells for foot disease for people wi...,NaN,"Crude rate of hospital spells, where the spell...","NHS England, multiple sources",16/02/2024,diabetes,\bdiabetes\b|\bhba1c\b|\bglyc\w*\b,diabetes,True
167,93712,IFCC-HbA1c <= 58 mmol/mol in patients with dia...,NaN,"The percentage of patients with diabetes, on t...",NHS England,10/12/2025,diabetes,\bdiabetes\b|\bhba1c\b|\bglyc\w*\b,diabetes,True
168,93713,IFCC-HbA1c <= 75 mmol/mol in patients with dia...,NaN,"The percentage of patients with diabetes, on t...",NHS England,10/12/2025,diabetes,\bdiabetes\b|\bhba1c\b|\bglyc\w*\b,diabetes,True
166,93711,Last BP in patients with diabetes is <= 140/80...,NaN,"The percentage of patients with diabetes, on t...",NHS England,10/12/2025,diabetes,\bdiabetes\b|\bhba1c\b|\bglyc\w*\b,diabetes,True
134,92287,Level of participation of GP practices in the ...,NaN,The percentage of practices that have returned...,"NHS England, National Diabetes Audit",25/11/2025,diabetes,\bdiabetes\b|\bhba1c\b|\bglyc\w*\b,diabetes,True
162,93047,Major lower-limb amputation procedures for peo...,NaN,Directly age and ethnicity standardised rate o...,"NHS England, multiple sources",16/02/2024,diabetes,\bdiabetes\b|\bhba1c\b|\bglyc\w*\b,diabetes,True


In [25]:
print("shortlist shape:", shortlist.shape)

shortlist shape: (501, 10)


In [26]:
shortlist.to_csv("_shortlist_fingertips_candidates.csv", index=False)
print("Saved:", "_shortlist_fingertips_candidates.csv")

Saved: _shortlist_fingertips_candidates.csv


In [27]:
check = pd.read_csv("_shortlist_fingertips_candidates.csv")
print("Reloaded shape:", check.shape)
check.head(30)

Reloaded shape: (501, 10)


,Indicator ID,Indicator,Simple Name,Definition,Data source,Date updated,match_theme,match_terms,primary_theme,keep
0,20101,Low birth weight of term babies,Low birth weight of term babies,Live births with a recorded birth weight under...,"OHID, based on Office for National Statistics ...",22/09/2025,maternity,\bmaternity\b|\bpregnan\w*\b|\bantenatal\b|\bp...,maternity,True
1,20401,Under 18s conception rate,Teenage pregnancy,"Conceptions in girls aged under 18 per 1,000 g...","OHID, based on Office for National Statistics ...",24/09/2025,maternity,\bmaternity\b|\bpregnan\w*\b|\bantenatal\b|\bp...,maternity,True
2,30315,Population vaccination coverage: Flu (at risk ...,NaN,Flu vaccine uptake (percent) in at risk indivi...,NHS England,07/10/2025,maternity,\bmaternity\b|\bpregnan\w*\b|\bantenatal\b|\bp...,maternity,True
3,41101,Emergency readmissions within 30 days of disch...,NaN,This indicator measures the percentage of emer...,NHS England,15/01/2026,maternity,\bmaternity\b|\bpregnan\w*\b|\bantenatal\b|\bp...,maternity,True
4,90282,Gap in the employment rate between those with ...,NaN,The percentage point gap between the percentag...,"OHID, based on Office for National Statistics ...",23/10/2023,maternity,\bmaternity\b|\bpregnan\w*\b|\bantenatal\b|\bp...,maternity,True
5,90283,Gap in the employment rate between those who a...,NaN,The percentage point gap between the percentag...,"OHID, based on NHS England and Office for Nati...",08/04/2024,maternity,\bmaternity\b|\bpregnan\w*\b|\bantenatal\b|\bp...,maternity,True
6,90362,Healthy life expectancy at birth,Healthy life expectancy at birth,A measure of the average number of years a per...,Office for National Statistics,21/07/2025,maternity,\bmaternity\b|\bpregnan\w*\b|\bantenatal\b|\bp...,maternity,True
7,90366,Life expectancy at birth,Life expectancy at birth,The average number of years a person would exp...,Office for National Statistics,26/01/2026,maternity,\bmaternity\b|\bpregnan\w*\b|\bantenatal\b|\bp...,maternity,True
8,90635,Gap in the employment rate for those who are i...,NaN,The percentage point gap between the percentag...,"OHID, based on NHS England and Office for Nati...",06/07/2022,maternity,\bmaternity\b|\bpregnan\w*\b|\bantenatal\b|\bp...,maternity,True
9,90740,"Ectopic pregnancy admissions rate / 100,000",NaN,Crude rate of ectopic pregnancy admissions to ...,"OHID, based on NHS England and Office for Nati...",07/02/2025,maternity,\bmaternity\b|\bpregnan\w*\b|\bantenatal\b|\bp...,maternity,True


## Output:
_shortlist_fingertips_candidates.csv
(the contract input for Notebook 02)
